# Produccion de Mexico 

In [31]:
import pandas as pd
# Leer archivo de producción de México
dfM = pd.read_csv(
    r'C:\Users\luisc\OneDrive\PIAV\RES\Produccion\Produccion_de_mexico.txt',
    sep='\t',               # separado por tabuladores
    skiprows=1,             # saltar primera fila vacía
    names=['Año', 'Cantidad Total (toneladas)', 'Valor en pesos'],
    thousands=',',          # manejar comas en números
    encoding='utf-8'
)

# Eliminar fila de TOTAL
dfM = dfM[dfM['Año'].notna()].copy()


# Convertir tipos
dfM['Año'] = dfM['Año'].astype(int)
dfM['Cantidad Total (toneladas)'] = pd.to_numeric(dfM['Cantidad Total (toneladas)'], errors='coerce')
dfM['Valor en pesos'] = pd.to_numeric(dfM['Valor en pesos'], errors='coerce')*1000

dfM_prod = dfM[['Año', 'Cantidad Total (toneladas)', 'Valor en pesos']].copy()

Tipo_cam = pd.read_excel('C:\\Users\\luisc\\OneDrive\\PIAV\\Tipo de cambio historico.xlsx')
Tipo_cam = Tipo_cam[['Fecha','SF63528']]
Tipo_cam['Fecha'] = Tipo_cam['Fecha'].astype(str).str[:4]
Tipo_cam =Tipo_cam.groupby('Fecha').agg(Tipo_Anual =('SF63528','mean')).reset_index()
Tipo_cam['Fecha'] = Tipo_cam['Fecha'].astype(int)
dfM_prod['Valor en pesos'] = dfM_prod['Valor en pesos'] / dfM_prod['Año'].map(Tipo_cam.set_index("Fecha")['Tipo_Anual'])
dfM_prod.rename(columns={'Valor en pesos': 'Valor Total (USD)'}, inplace=True)
dfM_prod.to_csv('RES_pro_MX.csv', index=False)


# Produccion de Canda

Encontrar codigos

In [34]:
import requests
import pandas as pd
from itertools import product as iproduct

# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURACIÓN — edita aquí
# ══════════════════════════════════════════════════════════════════════════════

# Palabras para buscar tablas
INCLUIR_TABLAS = [
    'beef', 'bovine', 'cattle', 'calf', 'calves',
    'cow', 'steer', 'heifer', 'slaughter', 'meat', 'carcass',
]

# Palabras para excluir tablas
EXCLUIR_TABLAS = [
    'index', 'indexes', 'price index', 'indicator',
]

# Palabras para filtrar miembros dentro de cada tabla
INCLUIR_SERIES = [
    'production', 'beef', 'bovine', 'cattle', 'calf', 'calves',
    'slaughter', 'meat', 'farm', 'output',
]

# Palabras para filtrar unidades de medida (solo estas pasan)
INCLUIR_UOM = [
    'dollar',
]

# Palabras para excluir unidades de medida
EXCLUIR_UOM = [
    'index', 
]

# ══════════════════════════════════════════════════════════════════════════════
# CÓDIGO — no necesitas tocar nada de aquí abajo
# ══════════════════════════════════════════════════════════════════════════════
N_DIMS    = 10
url_meta  = "https://www150.statcan.gc.ca/t1/wds/rest/getCubeMetadata"
url_coord = "https://www150.statcan.gc.ca/t1/wds/rest/getSeriesInfoFromCubePidCoord"
url_info  = "https://www150.statcan.gc.ca/t1/wds/rest/getSeriesInfoFromVector"

def contiene_alguna(texto, palabras):
    t = texto.lower()
    return any(p in t for p in palabras)

# ── 0. Catálogo de unidades ───────────────────────────────────────────────────
print('Cargando catálogo de unidades...')
codeSets    = requests.get("https://www150.statcan.gc.ca/t1/wds/rest/getCodeSets").json()['object']
uom_dict    = {n['memberUomCode']: n['memberUomEn'] for n in codeSets['uom'] if n.get('memberUomEn')}
scalar_dict = {n['scalarFactorCode']: n['scalarFactorDescEn'] for n in codeSets['scalar']}

uom_valor = {
    code: nombre for code, nombre in uom_dict.items()
    if     contiene_alguna(nombre, INCLUIR_UOM)
    and not contiene_alguna(nombre, EXCLUIR_UOM)
}
print(f'✅ {len(uom_valor)} unidades de valor:')
for code, nombre in sorted(uom_valor.items()):
    print(f'   {code:>4}  {nombre}')

# ── 1. Tablas relevantes ──────────────────────────────────────────────────────
print('\nDescargando tablas...')
tablas = pd.DataFrame(requests.get(
    "https://www150.statcan.gc.ca/t1/wds/rest/getAllCubesList",
    headers={'User-Agent': 'Mozilla/5.0'}
).json())

tablas_bovino = tablas[
     tablas['cubeTitleEn'].str.contains('|'.join(INCLUIR_TABLAS), case=False, na=False) &
    ~tablas['cubeTitleEn'].str.contains('|'.join(EXCLUIR_TABLAS), case=False, na=False)
][['productId', 'cubeTitleEn']].drop_duplicates()
print(f'✅ {len(tablas_bovino)} tablas:')
for _, row in tablas_bovino.iterrows():
    print(f"  {row['productId']} — {row['cubeTitleEn']}")

# ── 2. Series relevantes por tabla ────────────────────────────────────────────
todos_resultados = []

for _, tabla in tablas_bovino.iterrows():
    pid, title = tabla['productId'], tabla['cubeTitleEn']
    print(f'\n{"="*70}\n{pid} — {title}')

    data = requests.post(url_meta, json=[{"productId": pid}]).json()[0]
    if data.get("status") != "SUCCESS" or not data.get("object"):
        print('  ⚠️ Sin metadata'); continue

    meta        = data["object"]
    n_dims_real = len(meta["dimension"])
    listas_por_dim, saltar = [], False

    for dim in meta["dimension"]:
        pos = dim["dimensionPositionId"]

        if pos == 1:  # Geografía → solo Canada
            candidatos = [m for m in dim["member"] if m['memberNameEn'].strip().lower() == 'canada']
        else:         # Otras → INCLUIR_SERIES
            candidatos = [m for m in dim["member"] if contiene_alguna(m['memberNameEn'], INCLUIR_SERIES)]

        if not candidatos:
            print(f'  ⬛ Dim {pos} ({dim["dimensionNameEn"]}): sin candidatos')
            saltar = True; break

        listas_por_dim.append(candidatos)

    if saltar:
        continue


    for combo in iproduct(*listas_por_dim):
        ids  = [str(m['memberId']) for m in combo] + ['0'] * (N_DIMS - n_dims_real)
        coord = '.'.join(ids)

        obj = requests.post(url_coord, json=[{"productId": pid, "coordinate": coord}]).json()[0].get("object")
        if not obj: continue

        info = requests.post(url_info, json=[{"vectorId": obj["vectorId"]}]).json()[0].get("object")
        if not info: continue

        uom_code = info.get("memberUomCode", "")
        if uom_code not in uom_valor:
            print(f'No se incluira esta medida: {uom_code} | {uom_dict.get(uom_code, "")}');
            continue

        fila = {
            'productId'   : pid,
            'tableTitulo' : title,
            'coordinate'  : coord,
            'vectorId'    : obj["vectorId"],
            'titulo'      : info.get("SeriesTitleEn", ""),
            'scalarFactor': info.get("scalarFactorCode", ""),
            'scalarNombre': scalar_dict.get(info.get("scalarFactorCode", ""), ""),
            'uomCode'     : uom_code,
            'uomNombre'   : uom_valor[uom_code],
            'dims'        : ' | '.join(m['memberNameEn'] for m in combo),
        }
        todos_resultados.append(fila)
        print(f'  ✅ v{fila["vectorId"]} | {fila["scalarNombre"]} {fila["uomNombre"]} | {fila["titulo"][:50]}')

# ── 3. Guardar ────────────────────────────────────────────────────────────────
df_final = pd.DataFrame(todos_resultados)
print(f'\n{"="*70}\nTotal series: {len(df_final)}')
df_final.to_csv('series_bovino_canada.csv', index=False)
print('✅ Guardado en series_bovino_canada.csv')

Cargando catálogo de unidades...
✅ 102 unidades de valor:
      5  1992 constant dollars
      6  1992 constant dollars per square kilometre
     14  2002 constant dollars
     15  2002/2003 constant dollars
     18  2007 constant dollars
     23  2012 constant dollars
     47  Canadian dollars
     48  Canadian dollars per hundredweight
     49  Canadian dollars per unit of foreign currency
     60  Chained (2002) dollars
     61  Chained (2002) dollars per hour
     62  Chained (2007) dollars in thousands
     63  Chained (2007) dollars per hour
     75  Current dollars
     80  Dollar per 100 pound
     81  Dollars
     82  Dollars per 1.81 kilograms
     83  Dollars per 10 kilograms
     84  Dollars per 10 litres
     85  Dollars per 10 x 400 grams
     86  Dollars per 10,000 feet
     87  Dollars per 15 kilograms
     88  Dollars per 2 kilograms
     89  Dollars per 2.5 kilograms
     90  Dollars per 20 kilograms
     91  Dollars per 20 litres
     92  Dollars per 205 litres
     

\-Obtener cordendas

In [2]:
# ── 1. getCubeMetadata → encontrar memberIds de corn for grain + total farm value
url_meta = "https://www150.statcan.gc.ca/t1/wds/rest/getCubeMetadata"
r = requests.post(url_meta, json=[{"productId": 32100125}])
meta = r.json()[0]["object"]
for dim in meta["dimension"]:
    print(f"\n── Dim {dim['dimensionPositionId']}: {dim['dimensionNameEn']}")
    for m in dim["member"]:
             print(f"{m['memberId']:>4}  {m['memberNameEn']}")



── Dim 1: Geography
   1  Canada

── Dim 2: Livestock estimates
  13  Estimated farm output
   7  Total slaughter
   8  Inspected slaughter
   9  Uninspected slaughter
  10  Farm use, killed and eaten
  11  Sold, killed and sold
  12  Other uninspected slaughterings
  14  Live imports
  15  Live exports
  16  Average cold dressed weight
  18  Estimated meat production
  17  Edible offal

── Dim 3: Livestock
   1  Cattle
   2  Calves


In [26]:
# ── 2. getSeriesInfoFromCubePidCoordList → obtener el vectorId
# (ajusta la coordinate según lo que arroje el paso 1)
# Formato: "geo.crop.measure" → ej. "1.X.Y"

url_coord = "https://www150.statcan.gc.ca/t1/wds/rest/getSeriesInfoFromCubePidCoord"
body = [{"productId": 32100125, "coordinate": "1.18.1.0.0.0.0.0.0.0"}]  # <-- reemplaza X, Y

r2 = requests.post(url_coord, json=body)
obj = r2.json()[0]["object"]
print("coordinate:", obj["coordinate"])
print("vectorId  :", obj["vectorId"])
print("título    :", obj["SeriesTitleEn"])


vector_id = obj["vectorId"]

coordinate: 1.18.1.0.0.0.0.0.0.0
vectorId  : 1810283704
título    : Canada;Estimated meat production;Cattle


In [27]:
url_info = "https://www150.statcan.gc.ca/t1/wds/rest/getSeriesInfoFromVector"
r3 = requests.post(url_info, json=[{"vectorId": vector_id}])
info = r3.json()[0]["object"]

print("Título      :", info["SeriesTitleEn"])
print("scalarFactor:", info["scalarFactorCode"])
print("Unidad (UOM):", info["memberUomCode"])

Título      : Canada;Estimated meat production;Cattle
scalarFactor: 0
Unidad (UOM): 287


\- Escalar Factor y Unidad de medida

In [28]:
url_meta = "https://www150.statcan.gc.ca/t1/wds/rest/getCodeSets"
r = requests.get(url_meta)
meta = r.json()['object']
for n in meta['scalar']:
    if n['scalarFactorCode'] == 0:#Aqui lo buscas 
        print(f'\n{n['scalarFactorCode']}\n{n['scalarFactorDescEn']}\n{'-'*20}')
for n in meta['uom']:
    if n['memberUomCode'] == 287:
        print(f'\n{n['memberUomCode']}\n{n['memberUomEn']}\n{'-'*20}')    


0
units
--------------------

287
Tonnes
--------------------


In [33]:
# ── 4. getDataFromVectorsAndLatestNPeriods → datos históricos
url_data = "https://www150.statcan.gc.ca/t1/wds/rest/getDataFromVectorsAndLatestNPeriods"
r4 = requests.post(url_data, json=[{"vectorId": vector_id, "latestN": 60}])
puntos = r4.json()[0]["object"]["vectorDataPoint"]

df_corn_value_CA = (
    pd.DataFrame([{"Año": int(p["refPer"][:4]), "Cantidad Total (toneladas)": p["value"]}
                  for p in puntos])
    .sort_values("Año")
    .reset_index(drop=True)
)

df_corn_value_CA.dropna(inplace=True)
df_corn_value_CA.to_csv('RES_pro_CA.csv', index=False)

In [16]:
import pandas as pd
df_corn_value_CA = pd.read_csv('Produccion de Canada en dolares canadiences.csv')
dfCATons = pd.read_csv('ProduccionTonsCAN.csv')

dfCA = df_corn_value_CA.merge(dfCATons, on='Año')
dfCA = dfCA.drop(columns=[c for c in dfCA.columns if 'Unnamed' in c])
dfCA.to_csv('Maiz_Pro_CA.csv')

In [17]:
Tipo_CAD = pd.read_excel(r'C:\Users\luisc\OneDrive\PIAV\Tipo CAD.xlsx')
dfC_prod = pd.read_csv('Maiz_Pro_CA.csv')
Tipo_CAD = Tipo_CAD[['Fecha','Tipo']]
Tipo_CAD['Fecha'] = Tipo_CAD['Fecha'].astype(str).str[:4]
Tipo_CAD['Fecha'] = Tipo_CAD['Fecha'].astype(int)
dfC_prod['Valor Total (CAD)'] = dfC_prod['Valor Total (CAD)'] * dfC_prod['Año'].map(Tipo_CAD.set_index("Fecha")['Tipo'])
dfC_prod.rename(columns={'Valor Total (CAD)': 'Valor Total (USD)'}, inplace=True)
dfC_prod.drop(columns=[c for c in dfCA.columns if 'Unnamed' in c], inplace=True)
dfC_prod.to_csv('Maiz_Pro_CA.csv')

# Unated States of America Production 

In [25]:
import requests
import pandas as pd

API_KEY = "3F509C6D-D80C-354E-B64A-4DF483D7AF37"
BASE_URL = "https://quickstats.nass.usda.gov/api/api_GET/"

r = requests.get(BASE_URL, params={
    "key": API_KEY,
    "sector_desc": "ANIMALS & PRODUCTS",
    "statisticcat_desc": "PRODUCTION",
    "freq_desc": "ANNUAL",
    "agg_level_desc": "NATIONAL",
    "source_desc": "SURVEY",
    "format": "JSON"
})
df_meat_raw = pd.DataFrame(r.json()["data"])

# Cantidad: LB → toneladas métricas
df_qty = df_meat_raw[
    df_meat_raw["short_desc"] == "BEEF, SLAUGHTER - PRODUCTION, MEASURED IN LB"
][["year", "Value"]].copy()
df_qty["Value"] = pd.to_numeric(df_qty["Value"].str.replace(",", ""), errors="coerce")
df_qty["Value"] *= 0.000453592
df_qty.rename(columns={"year": "Año", "Value": "Cantidad Total (toneladas)"}, inplace=True)

# Valor USD: disponible bajo CATTLE, INCL CALVES (no existe "BEEF - PRODUCTION IN $")
df_usd = df_meat_raw[
    df_meat_raw["short_desc"] == "CATTLE, INCL CALVES - PRODUCTION, MEASURED IN $"
][["year", "Value"]].copy()
df_usd["Value"] = pd.to_numeric(df_usd["Value"].str.replace(",", ""), errors="coerce")
df_usd.rename(columns={"year": "Año", "Value": "Valor Total (USD)"}, inplace=True)

df_beef_us = (
    pd.merge(df_qty, df_usd, on="Año", how="outer")
    .sort_values("Año")
    .reset_index(drop=True)
    .dropna()
)

print(df_beef_us)
df_beef_us.to_csv("RES_Pro_USA.csv", index=False)
print("✅ Guardado: Beef_Pro_USA.csv")

     Año  Cantidad Total (toneladas)  Valor Total (USD)
59  1988                1.070024e+07       2.663692e+10
60  1989                1.047253e+07       2.707147e+10
61  1990                1.031604e+07       2.934824e+10
62  1991                1.039497e+07       2.939933e+10
63  1992                1.047162e+07       2.863252e+10
64  1993                1.045484e+07       2.884847e+10
65  1994                1.106129e+07       2.653358e+10
66  1995                1.144140e+07       2.469974e+10
67  1996                1.157884e+07       2.203493e+10
68  1997                1.156206e+07       2.494188e+10
69  1998                1.168453e+07       2.418755e+10
70  1999                1.201656e+07       2.609722e+10
71  2000                1.219573e+07       2.849867e+10
72  2001                1.189001e+07       2.940310e+10
73  2002                1.233453e+07       2.709753e+10
74  2003                1.194761e+07       3.211171e+10
75  2004                1.118059e+07       3.489